# SkillRadar – Del 4: Matematik, statistik og ranking
## Mundtlig eksamensforberedelse · Kildebaseret og med kørbare eksempler

Denne notebook forklarer **jeres faktiske implementering** fra `skillradar/demand.py`, `skillradar/report.py`, `skillradar/robustness.py`, `analysis/goal2.py` og de dokumenterede outputtal i `analysis/results/goal2_output.txt` (GitHub, gennemgået 5. november 2026).

**Læringsmål:** Forklar både *hvad* et mål betyder, *hvordan* det beregnes i Python, *hvorfor* I valgte det, *hvad outputs viser*, og *hvilke alternativer og begrænsninger* der findes.

**Bemærk:** De interaktive beregninger nedenfor er **syntetiske undervisningseksempler** og forudsætter ikke adgang til Kaggle. Kun tabeller tydeligt markeret **Faktisk SkillRadar-resultat** er tal fra jeres gemte analyse. Øvelserne kan køres igen uden at ændre projektfiler.

## 1. Sammenhængen mellem Del 1–4

- Del 1: Streamlit-appen tager et CV og en ønsket rolle.
- Del 2: Jobopslag bliver renset og omdannet til sæt af teknologier.
- Del 3: Keyword-matcheren finder O*NET-teknologier, og LLM/hybrid sammenlignes i evalueringen.
- **Del 4:** Vi tæller teknologier på tværs af relevante opslag og vurderer, hvor pålidelige rangeringerne er.

**Vigtigt skel:** Den enkelte `share`-værdi er et *deskriptivt estimat*. Konfidensintervaller, bootstrap og hypoteseprøver undersøger usikkerhed og robusthed under antagelser; de gør ikke udtrækket repræsentativt for det danske arbejdsmarked.

## 2. Notation, som du skal kunne til eksamen

Lad \(J_r\) være de relevante, deduplikerede opslag for rolle \(r\), og \(n_r=|J_r|\).

\(X_{it}=1\), hvis opslag \(i\) indeholder teknologi \(t\), ellers 0. Ét opslag kan nævne mange værktøjer, men hvert værktøj tælles højst én gang pr. opslag.

\[
\hat p_{t,r}=\frac{\sum_{i\in J_r}X_{it}}{n_r}=\frac{k_{t,r}}{n_r}
\]

`share` i koden svarer til \(\hat p\), `n_with_tool` svarer til \(k\), og `n_postings` er \(n\). `95% CI` er et interval for en underliggende andel *under en statistisk stikprøvemodel* – ikke et løfte om nutidens marked.

In [ ]:
from IPython.display import display
import numpy as np
import pandas as pd
import math
from collections import Counter

# Eksempel: fem fiktive, deduplikerede opslag for en rolle
postings = [
    {"Python", "SQL", "Docker"},
    {"Python", "SQL"},
    {"Python", "AWS"},
    {"SQL", "AWS"},
    {"Python"},
]
for i, tools in enumerate(postings, 1):
    print(f"Opslag {i}: {sorted(tools)}")

### Forklaring af output

Der er fem observationer, og hver observation indeholder et Python-`set`. Forskellige opslag kan indeholde de samme teknologier. Set-strukturen fjerner gentagelser *inden for* opslaget. **Alternativ:** optæl antal ordforekomster – men så måler man ikke længere »andelen af jobopslag, der kræver/omtaler teknologien«.

## 3. Demand: `demand_from_sets()` fra `skillradar/demand.py`

Jeres funktion konverterer `tool_sets` til en liste, tæller alle værktøjsnavne på tværs af sets med `value_counts()`, beregner `share = k/n` og tilføjer to Wilson-grænser. Den sorterer derefter efter `share` faldende og værktøjsnavn stigende ved lighed.

**Hvorfor?** Det er forståeligt, reproducerbart og let at sammenligne på tværs af CV og jobopslag. Det er ikke en machine-learning-prediktion.

In [ ]:
n = len(postings)
counts = Counter(tool for tool_set in postings for tool in tool_set)
demand = pd.DataFrame([
    {"tool": t, "n_with_tool": k, "n_postings": n, "share": k / n}
    for t, k in counts.items()
]).sort_values(["share", "tool"], ascending=[False, True]).reset_index(drop=True)
display(demand)
assert demand.set_index("tool").loc["Python", "share"] == 0.8

### Hvad viser outputtet?

Python findes i 4 ud af 5 opslag = 80 %, SQL i 3/5 = 60 %, Docker i 1/5 = 20 % og AWS i 2/5 = 40 %. Summen overstiger 100 %, fordi det samme opslag kan indeholde flere teknologier. At Python har 80 % demand betyder **ikke**, at 80 % af kandidaterne kan Python, eller at Python giver 80 % chance for ansættelse.

## 4. Procent, procentpoint og relative ændringer

Hvis junior-opslag nævner SQL i 36,2 % og senior-opslag i 66,2 %, er forskellen **30,0 procentpoint**:

\[
66.2\%-36.2\%=30.0\text{ procentpoint}.
\]

En relativ ændring ville være \((66.2-36.2)/36.2\approx82.9\%\). I bruger `diff_pp` = `(share_senior - share_junior) * 100`, **ikke** en relativ vækstprocent. Det er en kritisk sproglig forskel ved eksamen.

In [ ]:
junior, senior = .362, .662
print("Forskellen i procentpoint:", round((senior-junior)*100,1))
print("Relativt højere andel:", round((senior/junior-1)*100,1), "%")

## 5. Hvorfor 95 %-konfidensintervaller?

To andele kan være ens, men baseret på meget forskelligt antal opslag. 80 % ud af 10 er mere ustabilt end 80 % ud af 10.000. `demand.py` implementerer **Wilson score-interval** med \(z=1.96\), svarende til et nominelt 95 %-interval.

\[
\text{center}=\frac{\hat p+z^2/(2n)}{1+z^2/n},\quad
\text{half}=\frac{z\sqrt{\hat p(1-\hat p)/n+z^2/(4n^2)}}{1+z^2/n}
\]

Det returnerede interval er `[max(0, center-half), min(1, center+half)]`.

**Hvorfor Wilson frem for det almindelige Wald-interval?** Wilson er mere fornuftigt ved små stikprøver og andele tæt ved 0 og 1; Wald kan give dårlige grænser. **Alternativer:** Clopper–Pearson, Jeffreys eller bootstrap. For scrape-data gælder ingen af dem som en kur mod selektionsbias.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1 + z*z/n
    centre = (p + z*z/(2*n))/denom
    half = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/denom
    return max(0, centre-half), min(1, centre+half)

for k, n in [(8,10), (80,100), (800,1000), (0,10)]:
    lo, hi=wilson(k,n)
    print(f"{k}/{n}: {k/n:.1%} | Wilson 95 %: [{lo:.1%}, {hi:.1%}]")

### Hvad sker der i outputtet?

80 % får smallere intervaller, når antallet af opslag vokser. Ved 0/10 bliver intervallets øvre grænse større end nul, i modsætning til et naivt Wald-interval. **Eksamen:** Det er misvisende at sige, at der er 95 % sandsynlighed for, at det sande parameter ligger i *det allerede beregnede* interval. Det korrekte er, at metoden under sine antagelser har 95 % dækningsgrad over gentagne stikprøver.

## 6. `demand_table()` – sådan vælger I stikprøven

Fra koden: `filter_postings(pt, role, levels=None, country=None)` filtrerer på eksakt intern rolle, derefter eventuelle experience levels og eventuelt land. `demand_table` kalder `demand_from_sets(sub["tool_set"])` og gemmer `n_postings` og `role` i DataFrame-attributter.

**Hvorfor er filtre vigtige?** Alle tællere og nævnere er betinget af det valgte udsnit. Et procenttal uden `role`, `source`, `level`, `country` og `n` er ikke fuldt defineret.

**Nuance:** Appens »All experience levels« og »Entry & internships« bruger forskellige datasæt. Man kan ikke tilskrive alle forskelle alene erfaringstrinnet.

In [ ]:
small = pd.DataFrame([
    ("data analyst", "entry", "United States", {"SQL", "Excel"}),
    ("data analyst", "mid-senior", "United States", {"SQL", "Python"}),
    ("data analyst", "entry", "United States", {"Excel"}),
    ("software engineer", "entry", "United States", {"SQL", "Docker"}),
], columns=["role", "level", "country", "tool_set"])
sub = small[(small.role == "data analyst") & small.level.isin(["entry", "internship"])]
print("Indgår i junior data analyst:", len(sub))
print("SQL-andel:", sum("SQL" in x for x in sub.tool_set)/len(sub))

## 7. Coverage-score: hvordan et CV vurderes

Fra `skillradar/report.py` bruger `build_report()` kun værktøjer, der har `share >= min_share` (default 0,05) og derefter højst `top=15`. Den laver `on_cv = tool.isin(cv_tools)`. Formlen er:

\[
C=\frac{\sum_{t\in T_{15}}\hat p_t\,\mathbf 1(t\in CV)}{\sum_{t\in T_{15}}\hat p_t}.
\]

Hver teknologi vægtes efter hvor hyppig den er i jobopslagene. Det giver mere vægt til manglende Python (fx 78 %) end til et værktøj i 8 %.

**Vigtigt:** Coverage er **ikke** procentdel af jobs, brugeren er kvalificeret til, og ikke en ansættelsessandsynlighed. Det er en efterspørgselsvægtet dækning af et afgrænset topværktøjssæt.

In [ ]:
ranked = pd.DataFrame({"tool":["Python","SQL","Docker","AWS"],"share":[0.75,0.75,0.375,0.25]})
cv_tools={"Python","SQL"}
ranked["on_cv"] = ranked.tool.isin(cv_tools)
score=ranked.loc[ranked.on_cv,"share"].sum()/ranked.share.sum()
display(ranked)
print(f"Coverage: {score:.1%}")
assert round(score,4)==round(1.5/2.125,4)

### Sådan forklarer du outputtet

Python og SQL er fundet på CV'et og giver samlet vægt 1,50. Alle fire værktøjer vejer 2,125. Derfor 1,50 / 2,125 = **70,6 %**. Det betyder, at CV'et dækker 70,6 % af den *samlede demand-vægt* blandt disse fire værktøjer, ikke at CV'et matcher 70,6 % af jobopslagene.

## 8. Hvorfor `top=15` og `min_share=0.05`?

I appen bruger I `TOP=15`, og `build_report(..., top=TOP, min_share=0.05)`. Dermed undgår I at fremhæve meget sjældne teknologier. `min_share` anvendes **før** `head(top)`, og hvis der kun er 11 teknologier over 5 %, vil rapporten sammenligne 11, ikke 15.

**Alternativer:** En fast top-10-liste, threshold alene, samlet vægt i stedet for antal værktøjer, eller en indlæringsomkostningsjusteret ranking. Alle har tradeoffs: forskellige cutoffs kan ændre coverage for samme CV, og datadrevne thresholds kan gøre resultater sværere at sammenligne på tværs af roller.

In [ ]:
example = pd.DataFrame({"tool":["Python","SQL","Docker","SAS","OldTool"],
                        "share":[.75,.5,.12,.06,.02]})
for minimum in (.05,.10):
    chosen=example[example.share >= minimum].head(15)
    cv={"Python"}
    coverage=chosen.loc[chosen.tool.isin(cv),"share"].sum()/chosen.share.sum()
    print(f"min_share={minimum}: {list(chosen.tool)} | coverage {coverage:.1%}")

## 9. Rank er ikke det samme som demand-andel

`rank=1` betyder højest observeret `share` inden for den valgte sammenligning, men siger ikke hvor **meget** højere den ligger. Hvis to teknologier har 25,0 % og 24,9 %, er rækkefølgen mindre robust end ved 80 % og 10 %.

Derfor viser I både **shares, rang, 95 %-intervaller og bootstrap-rank-intervaller**. Et nominalt »nummer 7« uden usikkerhed kan skabe falsk præcision.

## 10. Bootstrap: Hvad er det? Hvorfor bruger I det?

Bootstrap er gentagen genudtagning **med tilbagelægning** fra de observerede opslag. I jeres `boot_shares(X, n_boot, rng)` er `X` en matrix med 0/1-værdier (opslag × værktøjer).

I bruger `rng.multinomial(n, np.full(n, 1/n), size=n_boot)` til at lave vægte, hvor hver genudtagning svarer til `n` opslag med tilbagelægning. Derefter udregnes `(W @ X) / n`.

**Hvorfor?** Den empiriske variation viser, om et værktøjs rang eller Spearman-korrelation ville svinge meget, hvis de samme datagenererende forhold gav en anden stikprøve.

**Tradeoff:** Det fanger stikprøveusikkerhed i det observerede datasæt, men ikke systematiske skævheder i scraping, rollefiltrering eller tekstmatcher.

In [ ]:
X = np.array([
    [1,1,0],[1,1,0],[1,0,1],[0,1,1],[1,0,0]
], dtype=np.int8)  # rækker=job, kolonner=Python/SQL/AWS
rng=np.random.default_rng(42)
n=len(X)
W=rng.multinomial(n, np.full(n, 1/n), size=8)
bootstrap_shares=(W@X)/n
print("Originale shares:", X.mean(axis=0))
print("Bootstrap shares (8 gentagelser):")
display(pd.DataFrame(bootstrap_shares,columns=["Python","SQL","AWS"]))
assert W.shape==(8,5)
assert (W.sum(axis=1)==5).all()

### Outputforklaring

Hver række er et nyt bootstrap-sample på fem opslag. Andelene varierer fra række til række. `W` er en effektiv måde at repræsentere, at nogle opslag bliver udtaget flere gange. Alternativ: vælg direkte indeks via `rng.choice(n, size=n, replace=True)` og beregn middelværdi. Multinomialvægte kan være effektive ved mange iterationer.

## 11. Rank-intervaller: Hvor stabil er plads nr. 1, 5 eller 14?

`rank_intervals()` vælger de `candidates=40` hyppigste værktøjer, bootstrapper deres shares og beregner en rang for hvert værktøj i hvert resample.

Kodelinjen `(-B).argsort(1).argsort(1) + 1` gør høj andel til lavt rangnummer (1 = højest). 2,5 %- og 97,5 %-kvantilerne af rank-fordelingen danner intervallet.

**Teknisk detalje:** Bootstrap-rangeringen bruger sortering; **ties** håndteres derfor ikke identisk med den viste punkt-rank (`rank(method="min")`). Rangintervallerne gælder desuden kun blandt de udvalgte top-40-kandidater, ikke hele værktøjskataloget.

**Fortolkning:** Ranginterval `1–1` er meget stabilt; `11–20` viser betydelig usikkerhed om det nøjagtige nummer.

In [ ]:
# Genskab principperne fra rank_intervals() på et lille dataset
rng=np.random.default_rng(7)
W=rng.multinomial(len(X),np.full(len(X),1/len(X)),size=2000)
B=(W@X)/len(X)
ranks=(-B).argsort(axis=1).argsort(axis=1)+1
rank_summary=pd.DataFrame({
    "tool":["Python","SQL","AWS"],
    "share":X.mean(axis=0),
    "rank_lo":np.quantile(ranks,.025,axis=0).astype(int),
    "rank_hi":np.quantile(ranks,.975,axis=0).astype(int)
})
display(rank_summary)

## 12. Faktiske SkillRadar-resultater: Rank-stabilitet for Data Scientist

Fra `analysis/results/goal2_output.txt`, **primary-datasættet med alle tilgængelige lande** (n=902):

| Rang | Teknologi | Demand | Bootstrap-ranginterval |
|---:|---|---:|---|
| 1 | Python | 78,2 % | 1–1 |
| 2 | SQL | 53,8 % | 2–2 |
| 3 | R | 48,8 % | 3–3 |
| 4 | AWS | 26,2 % | 4–4 |
| 5 | Tableau | 22,0 % | 5–6 |
| 6 | Spark | 21,0 % | 5–6 |
| 11 | SAS | 9,5 % | 10–15 |
| 13 | C++ | 7,9 % | 11–20 |

**Hvad betyder det?** I den analyserede stikprøve er Python sikkert nr. 1, men plads 13 for C++ er langt mere usikker. Den nøjagtige rækkefølge længere nede bør ikke behandles som objektiv sandhed om markedet.

**Vigtigt:** De 902 opslag her er ikke det samme udsnit som US-only-sammenligningen nedenfor (777); landeafgrænsningen er anderledes.

## 13. Replikation på tværs af to datasæt

I vil teste, om et værktøj, der ligger højt i primary, også ligger relativt højt i secondary. Funktionen `replication()`:

1. Filtrerer begge datasæt til samme rolle; i `analysis/goal2.py` sættes primary til `country_a="United States"` for bedre sammenlignelighed.
2. Tager **unionen** af top-20-værktøjer fra hver kilde, så værktøjer særligt vigtige i én kilde ikke forsvinder.
3. Opbygger en binær `tool_matrix` for hver kilde.
4. Beregner andele `share_a` og `share_b` for samtlige værktøjer.
5. Udregner **Spearman-korrelation** mellem de to sæt andele.
6. Bootstrapper opslagene separat i A og B og får et 95 %-interval for rho.

**Metodisk styrke:** Et positivt resultat i et separat scrape er bedre end kun at undersøge intern stabilitet i samme datasæt. **Begrænsning:** Det er stadig LinkedIn-baserede, ikke-repræsentative kilder med forskelle i population og indsamlingsmetode.

## 14. Spearman-rangkorrelation: Hvad betyder rho?

Spearman \(\rho\) måler, hvor stærkt to variable har en **monoton** sammenhæng i deres rangering. Jeres funktion rangtransformerer først `x` og `y` med pandas `rank()` og beregner så Pearson-korrelationen af rangene med `np.corrcoef`.

- \(\rho=1\): præcis samme rangorden.
- \(\rho=0\): ingen lineær korrelation mellem rangene.
- \(\rho=-1\): omvendt rangorden.

Ved ingen ties kan man bruge den kendte formel:
\[
\rho=1-\frac{6\sum_i d_i^2}{m(m^2-1)}
\]
men jeres implementering med gennemsnitsranks håndterer ties via pandas.

**Spearman er ikke en test af ens absolutte procenter:** To kilder kan have identisk rækkefølge, men væsentligt forskellige demands.

In [ ]:
def spearman_like_project(x,y):
    a,b=pd.Series(x,dtype=float),pd.Series(y,dtype=float)
    return float(np.corrcoef(a.rank(),b.rank())[0,1]) if len(a)>=3 and a.nunique()>=2 and b.nunique()>=2 else float('nan')

share_a=[.8,.6,.4,.2]
share_b=[.4,.3,.2,.1] # alle andele halveret
share_c=[.1,.2,.3,.4] # omvendt rækkefølge
print("Samme orden, andre niveauer:",spearman_like_project(share_a,share_b))
print("Omvendt orden:",spearman_like_project(share_a,share_c))

### Outputfortolkning

Første rho er 1, selvom alle procenter i B kun er halvt så store. Det viser, at høj replikation af **rangorden** ikke betyder, at datasættene måler identiske **niveauer**. Det andet rho er -1, fordi rangordenen er omvendt. Alternativ til Spearman: Kendall tau (god til ties) eller Pearson på shares (følsom for størrelsen af forskellene).

## 15. Faktiske SkillRadar-resultater: Replikation

Fra `analysis/results/goal2_output.txt`, sammenligning **A all levels** med primary filtreret til USA (top-20 fra hver kilde, union):

| Rolle | Primary (US) n | Secondary n | Værktøjer sammenlignet | Spearman ρ | Bootstrap 95 %-interval |
|---|---:|---:|---:|---:|---|
| Data scientist | 777 | 290 | 24 | 0,80 | 0,66–0,88 |
| Data analyst | 1.457 | 386 | 24 | 0,87 | 0,77–0,93 |
| Software engineer | 6.368 | 1.382 | 21 | 0,87 | 0,84–0,92 |
| ML engineer | 205 | 91 | 23 | 0,82 | 0,68–0,88 |
| UX designer | 411 | 137 | 26 | 0,63 | 0,46–0,72 |

**Fortolkning:** Rangordenen genfindes stærkere for data analyst og software engineer end for UX designer. ML-engineer-resultatet er markeret som *small sample* (secondary n=91). I skal forklare variation og datagrundlag frem for blot at sige, at metoden »virker for alle roller«.

## 16. Fairness-check: »levels matched«

I `analysis/goal2.py` laver I **to** replikationsanalyser:

- **A all levels:** Secondary omfatter alle niveauer; primary US-only.
- **B levels matched:** Secondary begrænses til `mid-senior` og `associate`, fordi primary ikke har et tilsvarende entry-level-udvalg.

Eksempel: Data scientist ændrer sig fra **ρ = 0,80** (A; 777 vs. 290) til **ρ = 0,76** (B; 777 vs. 126). Software engineer går fra **0,87** til **0,84**. Det understreger, at populationssammensætning kan påvirke rangordenen.

**Alternativ:** Match jobtitler, erfaring, region og brancher på mere detaljeret niveau eller anvend statistisk re-vægtning; men det kræver metadata af tilstrækkelig kvalitet.

## 17. Bootstrap 95 %-interval for Spearman

Fra `replication()` bruger I `boot_shares()` for begge dataset-matricer separat og beregner rho for hvert par af bootstrap-samples. I filtrerer `nan` fra og bruger 2,5 % og 97,5 % kvantiler.

**Hvorfor genudtage *opslag*, ikke individuelle (opslag, værktøj)-celler?** Værktøjer i samme opslag er afhængige. Opslaget er den naturlige observationsenhed; genudtagning af enkeltceller ville ødelægge den samvariation, der findes mellem værktøjer.

In [ ]:
# To uafhængige små posting-matricer (syntetiske)
A=np.array([[1,1,0],[1,0,0],[1,1,1],[0,1,0],[1,1,0],[0,1,1]],dtype=int)
B2=np.array([[1,1,0],[1,0,1],[1,1,0],[0,1,0],[1,0,0],[0,1,1]],dtype=int)
rng=np.random.default_rng(1)
def boot(X,reps):
    n=len(X); W=rng.multinomial(n,np.full(n,1/n),size=reps)
    return W@X/n
BA,BB=boot(A,1000),boot(B2,1000)
rs=[spearman_like_project(a,b) for a,b in zip(BA,BB)]
rs=np.array([r for r in rs if not np.isnan(r)])
print("Observeret rho:",round(spearman_like_project(A.mean(0),B2.mean(0)),3))
print("Bootstrap 95 %:",np.round(np.quantile(rs,[.025,.975]),3))
print("Gyldige bootstrap-rho:",len(rs))

### Outputfortolkning

Bootstrap-intervallet kan være meget bredt med kun seks opslag i hver gruppe og tre værktøjer. Det er et bevidst mini-eksempel. I jeres faktiske analyse med hundredvis/tusindvis af opslag bliver resultaterne normalt mere stabile. Bemærk, at udvælgelsen af top-værktøjer i `replication()` sker **før** bootstrap; hele selektionsprocessens usikkerhed er derfor ikke fuldt med i intervallet.

## 18. Sammenligning af entry-level og senior-annoncer

I `seniority()` analyseres *kun secondary* for at undgå at blande to scraperkilder sammen. I bruger:

- Junior: `entry` + `internship`.
- Senior: `mid-senior`.
- Værktøjslisten: union af begge gruppers top-20.
- `p1`: andel junioropslag med værktøjet.
- `p2`: andel senioropslag med værktøjet.
- `diff_pp = (p2-p1)*100`.
- To-proportion z-test med pooled standardfejl.
- Et 95 %-interval for **forskellen** med unpooled standardfejl.
- BH-korrektion for multipel testning og `significant = q < 0.05`.

**Hvorfor teste inden for samme datasæt?** Det reducerer forskelle i scraper, geografi og dataformat, som ellers kan forveksles med erfaringseffekt.

## 19. To-proportion z-test – matematikken i jeres kode

For junior \(n_1,p_1\) og senior \(n_2,p_2\) tester I \(H_0:p_1=p_2\) mod en tosidet alternativ hypotese.

Pooled andel:
\[
\hat p_{pool}=\frac{n_1p_1+n_2p_2}{n_1+n_2}
\]

Pooled standardfejl og teststatistik:
\[
SE_0=\sqrt{\hat p_{pool}(1-\hat p_{pool})(1/n_1+1/n_2)},\quad z=(p_2-p_1)/SE_0
\]

Tosidet p-værdi i koden: `math.erfc(abs(z)/sqrt(2))`.

I bruger derimod **unpooled** standardfejl til intervallet for forskellen:
\[
SE=\sqrt{p_1(1-p_1)/n_1+p_2(1-p_2)/n_2}
\]
\[
100\left[(p_2-p_1)\pm1.96SE\right]\text{ procentpoint}
\]

**Begrænsning:** Ved meget små samples eller værdier nær 0/1 er normalapproksimationen usikker. En test kan også blive påvirket af afhængighed mellem genopslag og variation i jobbeskrivelsernes længde.

In [ ]:
def compare_proportions(k1,n1,k2,n2):
    p1,p2=k1/n1,k2/n2
    pooled=(k1+k2)/(n1+n2)
    se0=math.sqrt(pooled*(1-pooled)*(1/n1+1/n2))
    z=(p2-p1)/se0 if se0 else 0.0
    pvalue=math.erfc(abs(z)/math.sqrt(2))
    se=math.sqrt(p1*(1-p1)/n1+p2*(1-p2)/n2)
    diff=(p2-p1)*100
    return {'junior_share':p1,'senior_share':p2,'diff_pp':diff,
            'z':z,'p':pvalue,'ci_low_pp':diff-1.96*se*100,'ci_high_pp':diff+1.96*se*100}

# Fiktivt eksempel, ikke projektets rå tællinger
print(compare_proportions(20,100,40,100))

### Outputforklaring

Her ses 20 % junior mod 40 % senior, altså +20 procentpoint. `z` og p-værdien tester nulhypotesen. Konfidensintervallet viser størrelsen af forskellen med usikkerhed. **Husk:** En lille p-værdi er ikke det samme som en stor praktisk effekt, og en stor effekt kan være usikker ved få opslag.

## 20. Hvorfor Benjamini–Hochberg (BH)?

I tester mange teknologier samtidig. Hvis man tester 25 uafhængige nulhypoteser på niveau 5 %, kan man få falske positive alene ved tilfældighed. `benjamini_hochberg()` justerer p-værdier til `q`-værdier, så beslutningerne kontrollerer **false discovery rate (FDR)** under almindelige BH-antagelser.

Den konkrete kode:

1. Sortér p-værdierne: `order = np.argsort(p)`.
2. Udregn `q = p_sorted * m / rank`.
3. Gør justeringerne monotone med kumulativt minimum **bagfra**.
4. Sæt øvre loft på 1 og genskab oprindelig rækkefølge.

**Hvorfor BH fremfor Bonferroni?** BH er mindre konservativ, når man udforsker mange sammenligninger, fordi målet er kontrol af forventet andel falske opdagelser, ikke den strengere family-wise error rate.

In [ ]:
def bh(p):
    p=np.asarray(p,dtype=float)
    m=len(p)
    if m==0:return p
    order=np.argsort(p)
    q=p[order]*m/np.arange(1,m+1)
    q=np.minimum.accumulate(q[::-1])[::-1]
    out=np.empty(m); out[order]=np.minimum(q,1)
    return out

pvals=np.array([.001,.020,.040,.15,.80])
print(pd.DataFrame({'p':pvals,'q_BH':bh(pvals),'significant_0_05':bh(pvals)<.05}))

### Outputfortolkning

En p-værdi under 0,05 er **ikke automatisk** signifikant efter BH-korrektion. `q < 0.05` er jeres faktiske kriterium. Det forklarer, hvorfor I ikke rapporterer alle tilsyneladende forskelle som sikre fund.

## 21. Faktisk SkillRadar-resultat: SQL hos Data Analyst

Fra `analysis/results/goal2_output.txt` — **secondary**, Data Analyst:

- Junior (entry/internship): **n=94**, SQL **36,2 %**.
- Senior (mid-senior): **n=148**, SQL **66,2 %**.
- `senior − junior`: **+30,0 procentpoint**.
- 95 %-interval: **+17,7 til +42,4 procentpoint**.
- BH-korrigeret `q` vises som **0,000** (afrundet, altså ikke matematisk nul).
- Markerede `significant = yes`.

Det er et af projektets tydeligste junior/senior-fund. Fortolkningen er **associationsbaseret**: Senioropslag i det valgte sample nævner SQL oftere. Det dokumenterer ikke, at senioritet *forårsager* højere SQL-efterspørgsel.

**Andre resultater:** Data Analyst R ligger 10,6 % vs. 24,3 %, men `q=0,075`; det er **ikke** signifikant efter jeres korrektion, selv om den ukorrigerede forskel er forholdsvis stor.

## 22. Faktisk SkillRadar-resultat: Software Engineer

Fra samme tabel, junior n=212 og senior n=725:

| Teknologi | Junior | Senior | Senior − junior | q | BH-signifikant? |
|---|---:|---:|---:|---:|---|
| SQL | 24,1 % | 31,0 % | +7,0 pp | 0,252 | Nej |
| GitLab | 7,1 % | 2,6 % | −4,5 pp | 0,024 | Ja |
| C++ | 30,2 % | 17,8 % | −12,4 pp | 0,002 | Ja |

**Hvorfor er resultatet interessant?** Flere værktøjer er *oftere nævnt i junioropslag*. Det viser, at man ikke kan antage, at seniorstillinger altid har højere efterspørgsel efter alle teknologier. Resultaterne kan også hænge sammen med forskelle i underroller, brancher og annoncepraksis.

## 23. Hvorfor »ikke signifikant« ikke betyder »ingen forskel«

Et resultat som Spark hos Data Scientist ligger ifølge jeres output på 7,9 % junior og 27,4 % senior, med en forskel på +19,5 pp, men BH `q=0,055`. Med jeres forudbestemte 0,05-grænse markeres det **ikke signifikant**.

**Forklaring:** Statistikken kan ikke udelukke en reel forskel, men giver heller ikke tilstrækkeligt grundlag til at kalde den statistisk sikker *efter multipel-test-korrektionen*. Med få opslag kan usikkerheden være stor. Vær opmærksom på, at de viste 95 %-intervaller for forskelle er **ukorrigerede**; deres udelukkelse af nul behøver derfor ikke give samme beslutning som BH-justeret q.

## 24. Konkrete forsigtighedspunkter i jeres kode

1. **Udvalgsskævhed:** Webscrapede LinkedIn-opslag er ikke nødvendigvis et tilfældigt eller repræsentativt udsnit af arbejdsmarkedet.
2. **Tekst-match-fejl:** Et værktøj kan nævnes uden at være et krav; matcheren kan have false positives og false negatives.
3. **Rollemapping:** Regex kan både overse og fejlklassificere titler.
4. **Dedup:** Identiske annoncer fjernes via teksthash inden for rolle; nær-dubletter kan stadig være med.
5. **Bootstrap-kandidater:** Rank-intervallerne er betinget af top-40-kandidaterne fra det observerede datasæt.
6. **Spearman:** Høj rho betyder samme orden, ikke samme niveau eller kausal sammenhæng.
7. **Mange tests:** BH-korrektion ændrer, hvilke fund der markeres signifikante.
8. **Små undergrupper:** UX/ML junior har meget få opslag; konklusioner herfra er usikre.
9. **Sammenligningsgrupper:** »All levels« og »entry-level« i appen bruger forskellige kildedatasæt.
10. **Snapshot:** Der kan ikke drages robuste påstande om stigende/faldende demand over tid.

## 25. Hvilke alternative metoder kunne I have brugt?

| Problem | Valgt metode | Alternativ | Hvorfor ikke nødvendigvis bedre? |
|---|---|---|---|
| Demand | Andel opslag med tool | Antal omtaler / TF-IDF | Mindre intuitiv jobopslagsandel |
| Interval for share | Wilson 95 % | Wald, eksakt binomial, Bayesian credible interval | Tradeoff mellem småsample-egenskaber og kompleksitet |
| Coverage | Demand-vægtet | Uvægtet Jaccard, vægt efter sværhedsgrad | Sværere at begrunde prioritering uden eksterne læringsomkostninger |
| Rank-stabilitet | Bootstrap på opslag | Analytisk rankmodel, Bayes ranking | Større modelantagelser |
| Sammenlign top-lister | Spearman | Kendall tau, Pearson, top-k overlap | Måler lidt forskellige aspekter af stabilitet |
| Junior/senior | To-proportion z-test | Fisher exact, logistisk regression, multilevel-model | Mere robuste alternativer kan kræve flere data og confounderfelter |
| Mange tests | Benjamini–Hochberg FDR | Bonferroni / Holm FWER | Strengere og færre fund |

**Til eksamen:** Beskriv både hvorfor den valgte metode er *passende for jeres scope* og de antagelser, der begrænser fortolkningen.

## 26. Hvad kan I sige om Goal 2?

Jer(es) formulering i README er: **»Measure current tool demand per role, and show it is robust.«** I skal dog tydeliggøre, at »current« henviser til tilgængelige data fra 2024, ikke jobmarkedet i november 2026.

**Dokumenteret støtte:**
- Spearman ρ 0,80–0,87 for data scientist, data analyst og software engineer i primær/sekundær-sammenligningen.
- Øverste værktøjer er ofte stabile i bootstrappede rank-intervaller.
- Senioritetsforskelle er generelt få efter BH-korrektion, med SQL hos data analysts som et klart eksempel.

**Dokumenterede undtagelser:** UX designer har lavere replikation (ρ=0,63), og ML engineer har et lille secondary-sample. Udtrækkene er ikke representative, og rank-robusthed siger ikke noget om ansættelseschancer.

## 27. Eksamensøvelse: Forklar et data-output i fire trin

Forestil dig, at eksaminator peger på »Python 78,2 %, rank 1–1« for Data Scientist:

1. **Definition:** Python blev fundet i 78,2 % af de 902 relevante, deduplikerede opslag i det pågældende udsnit.
2. **Metode:** Matcheren finder teknologien, og demand tæller den én gang pr. opslag.
3. **Usikkerhed:** Bootstrap over opslag viser, at den står nummer 1 i hele det viste 95 %-ranginterval; dette er ikke garanti om den virkelige markedsrang.
4. **Begrænsning:** Der kan være scraping-, matching- og rolleklassifikationsbias; kilden er historisk og mest international.

Gør derefter det samme for »SQL junior 36,2 % vs senior 66,2 %, q≈0«.

## 28. 18 spørgsmål til mundtlig eksamen

1. Hvad er forskellen på `n_with_tool` og `n_postings`?
2. Hvorfor kan demand-procenter summeres til over 100 %?
3. Hvorfor tæller I ikke flere omtaler af samme værktøj?
4. Hvordan er et Wilson score-interval anderledes end et Wald-interval?
5. Hvad betyder 95 % i konfidensintervallet?
6. Hvorfor er en høj coverage-score ikke sandsynligheden for at få job?
7. Hvad betyder `top=15` og `min_share=0.05`?
8. Hvad sker der med coverage, hvis threshold ændres?
9. Hvad er forskellen mellem demand share og rank?
10. Hvorfor bootstrapper I opslag og ikke enkelte værktøjsforekomster?
11. Hvad er et bootstrap rank-interval?
12. Hvorfor bruger I Spearman frem for kun at sammenligne demand-procenter?
13. Kan Spearman være 1 ved store forskelle i absolutte procenter?
14. Hvad betyder unionen af to top-20-lister?
15. Hvad er forskellen på p-værdi og BH-korrigeret q-værdi?
16. Hvad er nulhypotesen i junior/senior-testen?
17. Hvorfor kan et 95 %-interval udelukke 0, mens BH-testen ikke giver signifikans?
18. Hvilke begrænsninger forhindrer jer i at hævde, at I måler det danske jobmarked i dag?

## 29. Kort mundtligt eksamenssvar (ca. 2 minutter)

> »I SkillRadar definerer vi efterspørgsel som andelen af relevante jobopslag, der nævner en bestemt teknologi. Vi finder værktøjerne i de enkelte opslag og tæller hvert værktøj én gang pr. opslag. Det giver en procent, som en studerende kan forstå. Vi viser også et 95 % Wilson-konfidensinterval for at synliggøre stikprøveusikkerhed, især for roller med få opslag.
>
> Når vi sammenligner et CV med jobmarkedet, viser vi de højst rangerede værktøjer, som brugeren har og mangler. Coverage er efterspørgselsvægtet: At mangle Python, som nævnes ofte, vægter mere end at mangle et sjældent værktøj. Det er ikke en ansættelsessandsynlighed.
>
> Vi undersøger også om rangeringen er robust. Vi beregner Spearman-korrelation mellem to uafhængige LinkedIn-datasæt og bootstrapper rangeringerne for at se, hvilke topplaceringer der er stabile. For data scientist, data analyst og software engineer genfindes rækkefølgen relativt godt, mens UX-designer er mindre stabil.
>
> Endelig sammenligner vi entry-level og senioropslag inden for det samme datasæt. Vi bruger to-proportion z-test og korrigerer for mange sammenligninger med Benjamini–Hochberg. Eksempelvis nævnes SQL i 36,2 % af junior-dataanalystopslag og 66,2 % af senioropslag. Vi understreger, at kilderne er snapshots fra 2024 og ikke et repræsentativt billede af dagens danske arbejdsmarked.«

## 30. Kilder – direkte i jeres projekt

- [SkillRadar GitHub](https://github.com/Lucialululu/SkillRadar)
- [skillradar/demand.py](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/demand.py): `filter_postings`, `wilson`, `demand_from_sets`, `demand_table`
- [skillradar/report.py](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/report.py): `build_report` og coverage
- [skillradar/robustness.py](https://github.com/Lucialululu/SkillRadar/blob/main/skillradar/robustness.py): `tool_matrix`, `boot_shares`, `replication`, `rank_intervals`, `seniority`, `benjamini_hochberg`
- [analysis/goal2.py](https://github.com/Lucialululu/SkillRadar/blob/main/analysis/goal2.py): analyseflow, undergrupper, CSV/figurer
- [analysis/results/goal2_output.txt](https://github.com/Lucialululu/SkillRadar/blob/main/analysis/results/goal2_output.txt): de faktiske outputtal nævnt i notebooken
- [README.md](https://github.com/Lucialululu/SkillRadar/blob/main/README.md): mål, afgrænsning og samlede resultater

**Afgrænsning:** Når matematiske eksempler ikke er fra jeres faktiske projektresultater, er det angivet. Det forhindrer, at I ved en fejl præsenterer demonstrationer som empiriske fund.